<a href="https://colab.research.google.com/github/Rohitsingh24-cloud/legal-bert-based-contract-analyzer/blob/main/CONTRACT_ANALYZER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ----------------------------- CONFIG -----------------------------
CSV_PATH = "/content/master_clauses.csv"
OUT_DIR  = "/content/contract-analyzer"
MODEL_DIR = f"{OUT_DIR}/models/legal-bert-clf"
DATA_DIR  = f"{OUT_DIR}/data/clf"
INFER_CHECKPOINT = f"{MODEL_DIR}/checkpoint-117"
BASE_MODEL  = "nlpaueb/legal-bert-base-uncased"

# ----------------------------- install requireed library ------------------------------
import os, json, re, numpy as np, pandas as pd
from pathlib import Path
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)


df = pd.read_csv(CSV_PATH)

# find base columns that have a matching Answer

base_cols = []
for c in df.columns:
    if c.endswith("-Answer") or c.endswith("- Answer"):
        base = c.replace("-Answer", "").replace("- Answer", "").strip()
        if base in df.columns: base_cols.append(base)

seen, label_names = set(), []
for b in base_cols:
    if b not in seen:
        label_names.append(b); seen.add(b)

def _get_a_col(b, cols):
    for variant in (f"{b}-Answer", f"{b}- Answer"):
        if variant in cols: return variant
    return None

def row_to_text(row, clause_names):
    parts = []
    if "Document Name" in row and str(row["Document Name"]).strip():
        parts.append(f"Document Name: {str(row['Document Name']).strip()}")
    for b in clause_names:
        a_col = _get_a_col(b, df.columns)
        if a_col is None:
            continue
        val = row.get(a_col, "")
        if isinstance(val, float) and np.isnan(val): val = ""
        val = str(val).strip()
        if val: parts.append(f"{b}: {val}")
    return "\n".join(parts).strip()

def row_to_labels(row, clause_names):
    y = []
    for b in clause_names:
        a_col = _get_a_col(b, df.columns)
        val = "" if a_col is None else row.get(a_col, "")
        if isinstance(val, float) and np.isnan(val): val = ""
        y.append(1 if str(val).strip() else 0)
    return y

records = []
for _, r in df.iterrows():
    text = row_to_text(r, label_names)
    labels = row_to_labels(r, label_names)
    if any(labels):
        records.append({"text": text, "labels": labels})

prep_df = pd.DataFrame(records)

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(prep_df, test_size=0.2, random_state=42, shuffle=True)

def write_jsonl(_df, fp):
    with open(fp, "w", encoding="utf-8") as f:
        for _, row in _df.iterrows():
            f.write(json.dumps({"text": row["text"], "labels": row["labels"]}, ensure_ascii=False) + "\n")

write_jsonl(train_df, f"{DATA_DIR}/train.jsonl")
write_jsonl(val_df,   f"{DATA_DIR}/val.jsonl")
with open(f"{DATA_DIR}/labels.txt", "w", encoding="utf-8") as f:
    for n in label_names: f.write(n + "\n")

print("Prepared:", len(label_names), "labels;", len(train_df), "train;", len(val_df), "val")

# ============================ 2) train the model ============================
import inspect, transformers, torch
from transformers import TrainingArguments
print("Transformers version:", transformers.__version__)

def make_args(output_dir, lr=3e-5, train_bs=16, eval_bs=32, epochs=5, logging_steps=40):
    base = dict(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        logging_steps=logging_steps,
        fp16=torch.cuda.is_available(),
    )
    sig = inspect.signature(TrainingArguments.__init__)
    params = sig.parameters
    if "evaluation_strategy" in params and "save_strategy" in params:
        base.update(evaluation_strategy="epoch", save_strategy="epoch")
        if "load_best_model_at_end" in params: base["load_best_model_at_end"] = True
        if "metric_for_best_model" in params: base["metric_for_best_model"] = "macro_f1"
        if "greater_is_better" in params: base["greater_is_better"] = True
    else:
        if "do_eval" in params: base["do_eval"] = True
        if "eval_steps" in params: base["eval_steps"] = 500
        if "save_steps" in params: base["save_steps"] = 500
        for k in ["evaluation_strategy","save_strategy","load_best_model_at_end","metric_for_best_model","greater_is_better"]:
            base.pop(k, None)
    base = {k: v for k, v in base.items() if k in params}
    return TrainingArguments(**base)

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, Trainer
from torch import nn

num_labels = len(label_names)
tok = transformers.AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess(batch):
    enc = tok(batch["text"], truncation=True, padding="max_length", max_length=512)
    enc["labels"] = batch["labels"]
    return enc

ds = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl",
                                      "validation": f"{DATA_DIR}/val.jsonl"})
ds_tok = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)
ds_tok = ds_tok.with_format("torch")

class MultiLabelModel(nn.Module):
    def __init__(self, base_model_name, num_labels):
        super().__init__()
        self.base = AutoModel.from_pretrained(base_model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.base.config.hidden_size, num_labels)
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]  # CLS
        logits = self.classifier(self.dropout(pooled))
        loss = None
        if labels is not None:
            loss = nn.BCEWithLogitsLoss()(logits, labels.float())
        return {"loss": loss, "logits": logits}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1/(1+np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    tp = (preds & labels).sum(axis=0)
    fp = (preds & (1-labels)).sum(axis=0)
    fn = ((1-preds) & labels).sum(axis=0)
    f1_per_label = np.where((2*tp + fp + fn) > 0, (2*tp) / (2*tp + fp + fn), 0.0)
    macro_f1 = float(f1_per_label.mean()) if len(f1_per_label) else 0.0
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    micro_f1 = float((2*TP) / (2*TP + FP + FN + 1e-8))
    subset_acc = float((preds == labels).all(axis=1).mean())
    return {"macro_f1": macro_f1, "micro_f1": micro_f1, "subset_acc": subset_acc}

model = MultiLabelModel(BASE_MODEL, num_labels)

args = make_args(
    output_dir=MODEL_DIR,
    lr=3e-5,
    train_bs=16,
    eval_bs=32,
    epochs=5,
    logging_steps=40,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    tokenizer=tok,
    compute_metrics=compute_metrics
)

has_ckpt = any(p.name.startswith("checkpoint-") for p in Path(MODEL_DIR).glob("checkpoint-*"))
if not has_ckpt and not (Path(MODEL_DIR)/"pytorch_model.bin").exists() and not (Path(MODEL_DIR)/"model.safetensors").exists():
    trainer.train()
    trainer.save_model(MODEL_DIR)
    tok.save_pretrained(MODEL_DIR)
    with open(f"{MODEL_DIR}/labels.txt","w",encoding="utf-8") as f:
        f.write("\n".join(label_names))
    print("Saved:", MODEL_DIR)
else:
    print("Training skipped (weights/checkpoint exist).")

metrics = trainer.evaluate()
print("Validation metrics:", metrics)

# ----------------------------inferencing and extracting ---------------------------
import torch, torch.nn as nn
from typing import List, Tuple, Dict
from transformers import AutoTokenizer, AutoModel
from safetensors.torch import load_file as safe_load_file


DEFAULT_KEYS: Dict[str, List[str]] = {
    "Agreement Date": ["dated", "date of this agreement", "as of", "executed on"],
    "Effective Date": ["effective date", "commencement", "becomes effective"],
    "Expiration Date": ["expiration", "expire on", "term ends", "end date"],
    "Renewal Term": ["renew", "renewal", "auto-renew", "extend for", "additional term"],
    "Term": ["term shall", "initial term", "during the term"],
    "Termination": ["terminate", "termination", "for cause", "for convenience", "material breach"],
    "Governing Law": ["governing law", "laws of", "jurisdiction", "venue"],
    "Confidentiality": ["confidential", "non-disclosure", "confidentiality", "proprietary"],
    "Indemnity": ["indemnify", "indemnification", "hold harmless", "defend"],
    "Limitation Of Liability": ["limitation of liability", "shall not exceed", "cap on liability", "aggregate liability"],
    "Assignment": ["assign", "assignment", "may not assign", "no assignment"],
    "Change Of Control": ["change of control", "merger", "acquisition", "sale of substantially all assets"],
    "Payment": ["payment", "fees", "invoice", "payable", "due within"],
    "Audit Rights": ["audit", "inspect records", "books and records"],
    "Warranty": ["warranty", "warranties", "as is", "disclaims"],
    "Insurance": ["insurance", "insured", "certificate of insurance", "coverage"],
    "Third Party Beneficiary": ["third party beneficiary", "no third party beneficiaries"],
    "Notice": ["notice", "notices", "address for notice", "deliver notice", "written notice"],
}

# --- Aliases to map your project label names to canonical buckets  ---
ALIASES = {
    "Cap On Liability": "Limitation Of Liability",
    "Liability Cap": "Limitation Of Liability",
    "Limitation of Liability": "Limitation Of Liability",
    "Governing law": "Governing Law",
    "Notices": "Notice",
    "Effective date": "Effective Date",
    "Expiry": "Expiration Date",
    "Auto-Renewal": "Renewal Term",
    "Jurisdiction": "Governing Law",
}

# Load labels
labels_fp_model = os.path.join(INFER_CHECKPOINT, "labels.txt")
labels_fp = labels_fp_model if os.path.exists(labels_fp_model) else f"{DATA_DIR}/labels.txt"
with open(labels_fp, "r", encoding="utf-8") as f:
    LABELS = [ln.strip() for ln in f if ln.strip()]

# Build per-label keywords with canonical flag
COMMON_STOP = {
    "of","the","and","a","on","to","for","in","by","with","or","not","all",
    "any","each","every","per","as","at","be","is","are","was","were","an"
}
MIN_KW_LEN = 4

def _normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip()).lower()
def _token_set(s: str):
    return set(re.findall(r"[a-z0-9]+", s.lower()))

def build_label_keywords(all_labels, base_keys, aliases):
    alias_map = { _normalize(k): aliases[k] for k in aliases }
    keys_norm = { _normalize(k): v for k, v in base_keys.items() }
    out = {}
    for lab in all_labels:
        lab_norm = _normalize(lab)
        # exact / normalized
        if lab in base_keys:
            kws = base_keys[lab]; canonical = True
        elif lab_norm in keys_norm:
            orig = next(k for k in base_keys if _normalize(k) == lab_norm)
            kws, canonical = base_keys[orig], True

        elif lab in aliases and aliases[lab] in base_keys:
            kws, canonical = base_keys[aliases[lab]], True
        elif lab_norm in alias_map:
            canon = alias_map[lab_norm]
            canon_key = next((k for k in base_keys if _normalize(k) == _normalize(canon)), None)
            kws, canonical = (base_keys[canon_key], True) if canon_key else ([], False)
        else:

            lab_tok = _token_set(lab) - COMMON_STOP
            best_key, best_sc = None, -1
            for k in base_keys:
                sc = len(lab_tok & (_token_set(k) - COMMON_STOP))
                if sc > best_sc: best_key, best_sc = k, sc
            if best_key and best_sc >= 2:
                kws, canonical = base_keys[best_key], True
            else:
                kws, canonical = [], False

        pats = []
        for kw in kws:
            kwl = kw.strip().lower()
            if len(kwl) < MIN_KW_LEN or kwl in COMMON_STOP:
                continue
            pats.append(r"\b" + re.escape(kwl) + r"\b")
        out[lab] = {"kws": pats, "canonical": canonical}
    return out

KEYS_RESOLVED = build_label_keywords(LABELS, DEFAULT_KEYS, ALIASES)

# tokenizer
try:
    tok_inf = AutoTokenizer.from_pretrained(INFER_CHECKPOINT)
except Exception:
    try:
        tok_inf = AutoTokenizer.from_pretrained(MODEL_DIR)
    except Exception:
        tok_inf = AutoTokenizer.from_pretrained(BASE_MODEL)


class InferenceModel(nn.Module):
    def __init__(self, base_model_name, num_labels):
        super().__init__()
        self.base = AutoModel.from_pretrained(base_model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.base.config.hidden_size, num_labels)
    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = out.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(pooled))
        return logits

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
inf_model = InferenceModel(BASE_MODEL, len(LABELS))
st_path_safe = os.path.join(INFER_CHECKPOINT, "model.safetensors")
st_path_bin  = os.path.join(INFER_CHECKPOINT, "pytorch_model.bin")
if not os.path.exists(st_path_safe) and not os.path.exists(st_path_bin):
    st_path_safe = os.path.join(MODEL_DIR, "model.safetensors")
    st_path_bin  = os.path.join(MODEL_DIR, "pytorch_model.bin")

if os.path.exists(st_path_safe):
    from safetensors.torch import load_file as safe_load_file
    state = safe_load_file(st_path_safe, device="cpu")
elif os.path.exists(st_path_bin):
    state = torch.load(st_path_bin, map_location="cpu")
else:
    raise FileNotFoundError(f"No model weights found in {INFER_CHECKPOINT} or {MODEL_DIR}")

missing, unexpected = inf_model.load_state_dict(state, strict=False)
print("Loaded weights. Missing keys:", missing)
print("Unexpected keys:", unexpected)
inf_model.to(DEVICE).eval()

# ---- Sentence tokenizer & IMPORTANT sentence filter --------
_SENT_SPLIT = re.compile(
    r'(?<=[.!?;:])\s+|'
    r'[\r\n]+'
    r'|[•▪●]\s*'
    r'|(?<=\))\s+(?=[A-Z])'
)

def split_sentences(text: str) -> List[str]:
    text = re.sub(r'[ \t]+', ' ', text)
    sents = [s.strip(" -\u2022") for s in _SENT_SPLIT.split(text) if s and s.strip(" -\u2022")]
    return sents

# Generic cues
_RE_DATE = re.compile(r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|'
                      r'Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|'
                      r'Nov(?:ember)?|Dec(?:ember)?)[\s\-.,]*\d{1,2},\s*\d{4}\b|\b\d{1,2}/\d{1,2}/\d{2,4}\b', re.I)
_RE_MONEY = re.compile(r'\$\s?\d[\d,]*(?:\.\d+)?|\b\d+\s?(?:USD|dollars|INR|₹)\b', re.I)
_RE_PERCENT = re.compile(r'\b\d{1,3}\s?%\b')
_RE_DAYS = re.compile(r'\b\d{1,3}\s*(?:calendar\s+)?days?\b', re.I)
_RE_MUST_SHALL = re.compile(r'\b(shall|must|will)\b', re.I)

def _generic_hit(s: str) -> bool:
    return any(p.search(s) for p in (_RE_DATE, _RE_MONEY, _RE_PERCENT, _RE_DAYS, _RE_MUST_SHALL))

def filter_important_sentences_only(text: str) -> List[str]:
    """
    Returns ONLY important sentences (plain strings), ordered, unique.
    Importance = (canonical label keyword/name hit) OR (generic cues).
    """
    sents = split_sentences(text)
    keep, seen = [], set()

    for s in sents:
        ls = s.lower()
        useful = False

        # canonical labelname
        for lab, info in KEYS_RESOLVED.items():
            if not info["canonical"]:
                continue
            # label-name
            if re.search(r"\b" + re.escape(lab.lower()) + r"\b", ls, flags=re.IGNORECASE):
                useful = True; break
            # keywords
            if any(re.search(pat, ls, flags=re.IGNORECASE) for pat in info["kws"]):
                useful = True; break

        # generic cues
        if not useful and _generic_hit(s):
            useful = True

        if useful and s not in seen:
            keep.append(s.strip())
            seen.add(s)
    return keep

# ----------------------- Predictors ---------------------------
def predict_clause_presence(text: str, threshold: float = 0.5) -> Tuple[List[Tuple[str, float]], List[Tuple[str, float]]]:
    enc = tok_inf(text, truncation=True, padding="max_length", max_length=512, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = inf_model(**enc)
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    scored = sorted([(LABELS[i], float(probs[i])) for i in range(len(LABELS))], key=lambda x: -x[1])
    preds = [(l, p) for l, p in scored if p >= threshold]
    return scored, preds

def synthesize_answers_style(text: str, keys: Dict[str, List[str]] = None) -> str:

    keys = keys or DEFAULT_KEYS
    sents = split_sentences(text)
    lowers = [s.lower() for s in sents]
    out: Dict[str, str] = {}
    def score_sentence(ls, keywords): return sum(1 for k in keywords if k.lower() in ls)
    for clause, kws in keys.items():
        best_idx, best_score = -1, -1
        for i, ls in enumerate(lowers):
            sc = score_sentence(ls, kws)
            if sc > best_score:
                best_idx, best_score = i, sc
        if best_idx >= 0 and best_score > 0:
            out[clause] = sents[best_idx]
    return "\n".join(f"{k}: {v}" for k, v in out.items())

def predict_from_raw_contract(text: str, threshold: float = 0.5):
    answers_text = synthesize_answers_style(text, DEFAULT_KEYS)
    use_text = answers_text if answers_text.strip() else text
    return predict_clause_presence(use_text, threshold=threshold)

def make_report(text: str, threshold: float = 0.5, auto_extract: bool = True, top_k: int = 10):
    # classification
    if auto_extract:
        scored, preds = predict_from_raw_contract(text, threshold)
    else:
        scored, preds = predict_clause_presence(text, threshold)

    important_sentences = filter_important_sentences_only(text)

    return {
        "threshold": threshold,
        "auto_extract": auto_extract,
        "top_scores": [{"label": l, "score": s} for l, s in scored[:top_k]],
        "predicted": [{"label": l, "score": s} for l, s in preds],
        "important_sentences": important_sentences,
        "answers_style_used_for_ml": "\n".join(important_sentences),
        "input_preview": text[:800]
    }

# ============================ 4) GRADO UI ============================
!pip -q install gradio PyPDF2 python-docx

import gradio as gr, io
from PyPDF2 import PdfReader
from docx import Document

def _read_pdf(fp):
    reader = PdfReader(fp)
    return "\n".join(p.extract_text() or "" for p in reader.pages)

def _read_docx(fp):
    doc = Document(fp)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

def _read_txt(fp):
    data = fp.read()
    try:
        return data.decode("utf-8")
    except UnicodeDecodeError:
        return data.decode("latin-1", errors="ignore")

def analyze_contract_ui(file, text, threshold, auto_extract):
    if file is not None:
        ext = os.path.splitext(file.name)[1].lower()
        with open(file.name, "rb") as f:
            if ext == ".pdf": text = _read_pdf(f)
            elif ext == ".docx": text = _read_docx(f)
            elif ext == ".txt": text = _read_txt(f)
            else: return ("Unsupported file. Use .pdf, .docx, or .txt.", "", "", "",)
    text = (text or "").strip()
    if not text: return ("No input provided.", "", "", "",)

    rep = make_report(text, threshold=float(threshold), auto_extract=bool(auto_extract), top_k=10)

    preds = "\n".join(f"{x['label']:35s} {x['score']:.3f}" for x in rep["predicted"]) or "(none)"
    top10 = "\n".join(f"{x['label']:35s} {x['score']:.3f}" for x in rep["top_scores"]) or "(none)"

    important_joined = "\n".join(rep.get("important_sentences", [])) or "(none)"
    full_json = json.dumps({"important_sentences": rep.get("important_sentences", [])}, indent=2)
    return preds, top10, important_joined, full_json

with gr.Blocks() as demo:
    gr.Markdown("## Contract Analyzer — Important Sentences Only")
    with gr.Row():
        file = gr.File(label="Upload contract (.pdf / .docx / .txt)", file_types=[".pdf", ".docx", ".txt"])
        text = gr.Textbox(lines=10, label="Or paste contract text")
    th = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Model Threshold (for scores)")
    auto = gr.Checkbox(value=True, label="Use sentence heuristics for model input")
    btn = gr.Button("Analyze")

    out_preds = gr.Textbox(label="Predicted clauses (≥ threshold)")
    out_top   = gr.Textbox(label="Top 10 scores")
    out_ans   = gr.Textbox(label="Important sentences (only)", lines=10)
    out_json  = gr.Textbox(label="JSON (only important_sentences list)", lines=14)

    btn.click(analyze_contract_ui, [file, text, th, auto], [out_preds, out_top, out_ans, out_json])

demo.launch(share=True)



Prepared: 41 labels; 408 train; 102 val
Transformers version: 4.57.1


/tmp/ipython-input-128977716.py:171: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training skipped (weights/checkpoint exist).


Validation metrics: {'eval_loss': 0.714735209941864, 'eval_model_preparation_time': 0.0047, 'eval_macro_f1': 0.5190549619499901, 'eval_micro_f1': 0.6598696306189874, 'eval_subset_acc': 0.0, 'eval_runtime': 0.8133, 'eval_samples_per_second': 125.41, 'eval_steps_per_second': 4.918}
Loaded weights. Missing keys: []
Unexpected keys: []
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f93f9caf415f2c55c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
